In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. 配置全局嵌入模型
Settings.embed_model = HuggingFaceEmbedding("BAAI/bge-small-zh-v1.5")

# 2. 创建示例文档
texts = [
    "张三是法外狂徒",
    "LlamaIndex是一个用于构建和查询私有或领域特定数据的框架。",
    "它提供了数据连接、索引和查询接口等工具。"
]
docs = [Document(text=t) for t in texts]

# 3. 创建索引并持久化到本地
index = VectorStoreIndex.from_documents(docs)
persist_path = "./llamaindex_index_store"
index.storage_context.persist(persist_dir=persist_path)
print(f"LlamaIndex 索引已保存至: {persist_path}")


In [6]:
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. 必须配置与创建索引时完全相同的嵌入模型
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-zh-v1.5")

# 2. 从本地加载索引
persist_path = "./llamaindex_index_store"
storage_context = StorageContext.from_defaults(persist_dir=persist_path)
index = load_index_from_storage(storage_context)

# 3. 将索引转换为检索器 (Retriever)
# 注意：不要使用 index.as_query_engine()，因为那会调用 LLM
retriever = index.as_retriever(similarity_top_k=1)

# 4. 执行相似度匹配
query = "张三是做什么的？"
nodes = retriever.retrieve(query)

# 5. 打印结果
print(f"\n查询: '{query}'")
print("相似度最高的文档片段:")
for node in nodes:
    # node.node.text 包含原始文本
    # node.score 包含相似度分数
    print(f"- 内容: {node.node.get_content()}")
    print(f"- 分数: {node.score:.4f}")

Loading llama_index.core.storage.kvstore.simple_kvstore from ./llamaindex_index_store\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from ./llamaindex_index_store\index_store.json.

查询: '张三是做什么的？'
相似度最高的文档片段:
- 内容: 张三是法外狂徒
- 分数: 0.6030
